# Differential Expression (limma-trend) - headless\n\nReproduces a paper's finding from a deposited matrix of ALREADY-NORMALIZED values\n(TPM, FPKM, CPM or similar). Use the DESeq2 templates for integer counts.\n

In [ ]:
# Parameters (injected at launch). All are str/number so injection stays valid R.
counts_path <- "/data/matrix.tsv"           # a deposited matrix of ALREADY-NORMALIZED values
output_path <- "/outputs/de_results.csv"
id_column <- ""                               # the deposit's own id column; "" means the first column
test_samples <- ""                            # comma-separated column names (treatment)
reference_samples <- ""                       # comma-separated column names (control)
block_labels <- ""                            # optional matched-pairs, ALIGNED to c(test,reference)
technical_group_labels <- ""                  # change_7.5 section 3.3: per-column technical group, ALIGNED to
                                              # c(test,reference); an ungrouped column carries its own name
lfc_threshold <- 1.0
padj_threshold <- 0.05
already_logged <- "false"                     # "true" when step 6 measured log_transformed values
\n

In [ ]:
suppressMessages(library(limma))

# limma-trend on log2 values. This template exists because the other three headless reproducers are
# DESeq2, which requires integer counts and estimates its own size factors: handing it TPM or CPM
# invalidates the dispersion model and returns numbers that are confidently wrong rather than
# obviously wrong. The bioinformaticians who asked for the deposit route said plainly that a TPM
# table is a thing we will be handed, and GSE274331's deposit is exactly one.
#
# limma-trend is the standard, defensible choice for a matrix that is already normalized: it fits a
# linear model per feature and lets the mean-variance trend absorb the fact that we did not start
# from counts. It is NOT equivalent to what a paper running DESeq2 on raw counts did, and the
# reproduction bundle records `method: limma_trend` for exactly that reason, so a divergence can be
# attributed to the method rather than charged to the paper.

test_s <- trimws(strsplit(test_samples, ",")[[1]]); test_s <- test_s[test_s != ""]
ref_s  <- trimws(strsplit(reference_samples, ",")[[1]]); ref_s <- ref_s[ref_s != ""]
stopifnot(length(test_s) > 0, length(ref_s) > 0)
samples <- c(test_s, ref_s)
condition <- factor(c(rep("test", length(test_s)), rep("reference", length(ref_s))), levels = c("reference", "test"))

mat <- read.delim(counts_path, check.names = FALSE, stringsAsFactors = FALSE)
# A deposited matrix often leaves its id column unnamed (GSE274331 does), so "" means "the first
# column" rather than an error. read.delim names such a column X, which is why the fallback is
# positional rather than by name.
if (!nzchar(id_column) || !(id_column %in% colnames(mat))) id_column <- colnames(mat)[1]
rownames(mat) <- make.unique(as.character(mat[[id_column]]))

missing <- setdiff(samples, colnames(mat))
if (length(missing) > 0) stop(paste("samples not in matrix:", paste(missing, collapse=", ")))
vals <- as.matrix(mat[, samples, drop = FALSE])
mode(vals) <- "numeric"

# change_7.5 section 3.3: collapse ONLY within a technical group (repeated measurements of one
# biological sample, confirmed by evidence), by averaging on the scale the values are stored in, so
# before any log is taken. Every column carries a label; an ungrouped one its own name. The block
# labels are collapsed with their columns.
if (!exists("technical_group_labels")) technical_group_labels <- ""
tg_s <- trimws(strsplit(technical_group_labels, ",", fixed = TRUE)[[1]])
if (length(tg_s) == length(samples) && length(unique(tg_s)) < length(tg_s)) {
  groups <- unique(tg_s)
  first <- match(groups, tg_s)
  vals <- sapply(groups, function(g) rowMeans(vals[, tg_s == g, drop = FALSE]))
  vals <- matrix(vals, ncol = length(groups), dimnames = list(rownames(mat), groups))
  condition <- condition[first]
  if (exists("block_labels") && nzchar(block_labels)) {
    bl <- trimws(strsplit(block_labels, ",", fixed = TRUE)[[1]])
    if (length(bl) == length(samples)) block_labels <- paste(bl[first], collapse = ",")
  }
  samples <- groups
  cat("collapsed", length(tg_s), "columns into", length(groups), "biological samples by technical group\n")
}

# Log2 once, and only if step 6 did not already measure the deposit as log-transformed. Logging a
# log would compress real differences into nothing and quietly produce a null result.
if (tolower(already_logged) != "true") {
  vals <- log2(vals + 1)
}

# Features with no signal carry no information and destabilise the variance trend.
keep <- rowSums(is.finite(vals)) == ncol(vals) & apply(vals, 1, function(r) any(r > 0))
vals <- vals[keep, , drop = FALSE]
if (nrow(vals) == 0) stop("no features with signal remained after filtering the deposited matrix")

if (!exists("block_labels")) block_labels <- ""
block_s <- trimws(strsplit(block_labels, ",")[[1]]); block_s <- block_s[block_s != ""]
use_block <- length(block_s) == length(samples) && length(unique(block_s)) >= 2
if (use_block) {
  block <- factor(block_s)
  design <- model.matrix(~ block + condition)
} else {
  design <- model.matrix(~ condition)
}

fit <- lmFit(vals, design)
# trend=TRUE is the "limma-trend" part: it models the mean-variance relationship that survives in
# normalized data, which is what makes this defensible on a matrix we did not normalize ourselves.
fit <- eBayes(fit, trend = TRUE)
coef_name <- tail(colnames(design), 1)
res <- topTable(fit, coef = coef_name, number = Inf, sort.by = "none")

# Same output contract as the DESeq2 templates (id, log2FoldChange, pvalue, padj), so _extract_reproduced_set
# and the concordance service need no change and the two routes stay comparable.
dir.create(dirname(output_path), showWarnings = FALSE, recursive = TRUE)
out <- data.frame(gene_id = rownames(res), log2FoldChange = res$logFC, pvalue = res$P.Value, padj = res$adj.P.Val)
write.csv(out, output_path, row.names = FALSE)
cat("wrote", nrow(out), "features to", output_path, "\n")
cat("method: limma-trend on", if (tolower(already_logged) == "true") "pre-logged" else "log2(x+1)", "values\n")
if (use_block) cat("design: ~ block + condition (paired,", length(unique(block_s)), "subjects)\n")
